# H/k Agent: BrowserGym WebArena-Verified Hard

Dieses Notebook steuert die neue saubere H/k-Architektur unter `scripts/hk_agent/`. Es nutzt den WebArena-Verified Hard Subset mit 258 Tasks und startet BrowserGym-Tasks ueber IDs wie `browsergym/webarena_verified.{intent_template_id}.{task_id}.{revision}`.

Die Runtime-Metriken dienen nur Replanning und Prozessanalyse. Offizieller Erfolg kommt aus WebArena-Verified `eval-tasks`.

In [20]:
from pathlib import Path
import json
import os
import subprocess
import shutil
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RUN_ROOT = ROOT / 'runs' / 'hk-agent'
OFFICIAL_REPO = ROOT / 'external' / 'webarena-verified'
sys.path.insert(0, str(ROOT / 'scripts'))
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_columns', 100)
ROOT


PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code')

## Environment Check

In [21]:
check_cmd = ['uv', 'run', 'python', '-c', "import gymnasium as gym; import browsergym.webarena_verified; ids=[s.id for s in gym.envs.registry.values() if s.id.startswith('browsergym/webarena_verified')]; print(len(ids)); print(ids[:3])"]
subprocess.run(check_cmd, cwd=ROOT, check=True)


812
['browsergym/webarena_verified.279.0.2', 'browsergym/webarena_verified.279.1.2', 'browsergym/webarena_verified.279.2.2']


CompletedProcess(args=['uv', 'run', 'python', '-c', "import gymnasium as gym; import browsergym.webarena_verified; ids=[s.id for s in gym.envs.registry.values() if s.id.startswith('browsergym/webarena_verified')]; print(len(ids)); print(ids[:3])"], returncode=0)

## Smoke Configuration

In [22]:
RUN_SMOKE = False

# Gemma 4 setup:
# - 26B A4B as planner for stronger long-horizon reasoning.
# - E4B as executor for cheaper/faster action grounding.
PLANNER_MODEL = 'gemma4:26b'
EXECUTOR_MODEL = 'gemma4:e4b'
EXPERIMENT_NAME = 'hk-agent-browsergym-random-gemma4-smoke'

# Random stratified smoke. WebArena uses GitLab, not GitHub.
USE_RANDOM_SAMPLE = True
RANDOM_SAMPLE_SEED = 42  # Set to None for a fresh random sample on every run.
SAMPLE_SITES = ['gitlab', 'reddit', 'shopping', 'shopping_admin']
SAMPLE_BUCKETS = ['short', 'medium', 'long']
SAMPLE_TASK_TYPES = None  # Example: ['NAVIGATE', 'RETRIEVE'] for a less destructive smoke.
SAMPLE_PER_GROUP = 1
EXCLUDE_TASK_IDS = {44, 157, 105, 27, 118}

# Manual fallback examples if USE_RANDOM_SAMPLE = False.
MANUAL_SMOKE_TASKS = ['44', '157', '105', '27', '118']
SMOKE_TASKS = MANUAL_SMOKE_TASKS.copy()

SMOKE_HS = ['0', '2', '5']
# k=0 is the no-periodic-runtime-validation baseline.
SMOKE_KS = ['0', '2', '5']

# Optional cleanup is intentionally disabled by default because it deletes run artifacts.
CLEAN_PREVIOUS_EXPERIMENT = False

# Set these to your chosen price/proxy per 1k tokens. For local Ollama the
# direct API cost is 0, but nonzero values are useful for thesis trade-off plots.
PLANNER_COST_PER_1K = 0.0
EXECUTOR_COST_PER_1K = 0.0
LAMBDA_COST = 1.0
LAMBDA_RUNTIME_SECONDS = 0.0

SMOKE_ENV = {
    'PYTHONUNBUFFERED': '1',
}


## Task Membership Check

In [23]:
from hk_agent.task_loader import (
    build_gym_id,
    infer_official_task_type,
    intent_bucket,
    sample_tasks_by_site_and_bucket,
)

dataset = json.loads((OFFICIAL_REPO / 'assets/dataset/webarena-verified.json').read_text())
tasks_by_id = {int(task['task_id']): task for task in dataset}
hard_ids = set(json.loads((OFFICIAL_REPO / 'assets/dataset/subsets/webarena-verified-hard.json').read_text())['task_ids'])

def draw_random_smoke_tasks(seed=RANDOM_SAMPLE_SEED):
    sampled = sample_tasks_by_site_and_bucket(
        OFFICIAL_REPO,
        sites=SAMPLE_SITES,
        buckets=SAMPLE_BUCKETS,
        task_types=SAMPLE_TASK_TYPES,
        per_group=SAMPLE_PER_GROUP,
        seed=seed,
        hard_only=True,
        single_site_only=True,
        supported_sites_only=True,
        exclude_task_ids=set(EXCLUDE_TASK_IDS),
    )
    return [str(task.task_id) for task in sampled]

if USE_RANDOM_SAMPLE:
    SMOKE_TASKS = draw_random_smoke_tasks()

missing_tasks = [task_id for task_id in SMOKE_TASKS if int(task_id) not in tasks_by_id]
if missing_tasks:
    raise ValueError(f'Unknown task ids: {missing_tasks}')

task_meta_rows = []
for raw_id in SMOKE_TASKS:
    task_id = int(raw_id)
    task = tasks_by_id[task_id]
    task_intent = task.get('intent', '')
    task_meta_rows.append({
        'task_id': task_id,
        'hard_subset': task_id in hard_ids,
        'site': ','.join(task.get('sites', [])),
        'task_type': infer_official_task_type(task),
        'intent_length': len(task_intent),
        'intent_bucket': intent_bucket(task_intent),
        'intent_template_id': task.get('intent_template_id'),
        'revision': task.get('revision'),
        'gym_id': build_gym_id(task),
        'intent': task_intent,
    })

task_meta_df = pd.DataFrame(task_meta_rows)
ALLOW_NON_HARD_TASK_IDS = not task_meta_df['hard_subset'].all()
display(task_meta_df.sort_values(['site', 'intent_bucket', 'task_id']))
print('Selected task ids:', ' '.join(SMOKE_TASKS))
print('Hard subset tasks:', int(task_meta_df['hard_subset'].sum()), '/', len(task_meta_df))


,task_id,hard_subset,site,task_type,intent_length,intent_bucket,intent_template_id,revision,gym_id,intent
2,444,True,gitlab,MUTATE,205,long,308,2,browsergym/webarena_verified.308.444.2,Update and commit (to a new branch called title-update with no merged request) the website code for the current project using the simple online file editor to change the browse...
1,800,True,gitlab,MUTATE,95,medium,600,2,browsergym/webarena_verified.600.800.2,"create a new group ""x-lab"" with members JonasVautherin, dilipchandima, dawiss1337, bmyun, DCMJY"
0,522,True,gitlab,MUTATE,29,short,352,2,browsergym/webarena_verified.352.522.2,Fork all repos from facebook.
5,28,True,reddit,RETRIEVE,267,long,33,2,browsergym/webarena_verified.33.28.2,"In the Worcester forum, get the username and post title of the most recent post, and count the number of comments on that post that are not from the author and have more downvo..."
4,644,True,reddit,MUTATE,134,medium,16,2,browsergym/webarena_verified.16.644.2,"Post a notice in games forum titled ""Tears of Kingdom Meet up!"". Set post details to ""virtual meetup for Tears of Kingdom on Dec 15th"""
3,407,True,reddit,MUTATE,49,short,22,2,browsergym/webarena_verified.22.407.2,Upvote the newest post in the deep learning forum
8,507,True,shopping,MUTATE,140,long,172,2,browsergym/webarena_verified.172.507.2,Buy the highest rated product from the Ceiling light category within a budget above 1000. Discard any items in your cart if it is not empty.
7,795,True,shopping,MUTATE,99,medium,191,2,browsergym/webarena_verified.191.795.2,"Change the delivery address for my second most recent order to 6726 McPherson Blvd, Pittsburgh, PA."
6,387,True,shopping,RETRIEVE,50,short,1356,2,browsergym/webarena_verified.1356.387.2,Who gave 4 or 5 stars for phone cases from EYZUTAK
11,108,True,shopping_admin,RETRIEVE,207,long,270,2,browsergym/webarena_verified.270.108.2,"Get the monthly count of completed orders from January 2023 through May 2023, inclusive. Return a list of objects with keys ""month"" (month name) and ""count"" (as integer) only, ..."


Selected task ids: 522 800 444 407 644 28 387 795 507 505 15 108
Hard subset tasks: 12 / 12


## Example Dataset Rows

In [24]:
all_task_rows = []
for task in dataset:
    task_id = int(task['task_id'])
    task_intent = task.get('intent', '')
    all_task_rows.append({
        'task_id': task_id,
        'hard_subset': task_id in hard_ids,
        'site': ','.join(task.get('sites', [])),
        'task_type': infer_official_task_type(task),
        'intent_bucket': intent_bucket(task_intent),
        'intent_length': len(task_intent),
        'intent_template_id': task.get('intent_template_id'),
        'revision': task.get('revision'),
        'gym_id': build_gym_id(task),
        'intent': task_intent,
    })

all_tasks_df = pd.DataFrame(all_task_rows)
hard_overview = all_tasks_df[all_tasks_df['hard_subset']].groupby(['site', 'task_type', 'intent_bucket'], dropna=False).agg(
    tasks=('task_id', 'count'),
    example_task_id=('task_id', 'first'),
    example_intent=('intent', 'first'),
).sort_values(['site', 'task_type', 'intent_bucket'])
display(hard_overview)

selected_examples = task_meta_df[['task_id', 'hard_subset', 'site', 'task_type', 'intent_bucket', 'gym_id', 'intent']].copy()
display(selected_examples.sort_values(['site', 'intent_bucket', 'task_id']))

example_rows = all_tasks_df[all_tasks_df['hard_subset']].groupby(['site', 'task_type'], dropna=False).head(2)
display(example_rows[['task_id', 'site', 'task_type', 'intent_bucket', 'gym_id', 'intent']].sort_values(['site', 'task_type', 'task_id']).head(20))


tasks  example_task_id  \
site               task_type intent_bucket                           
gitlab             MUTATE    long              12              415   
                             medium            15              446   
                             short              9              397   
                   NAVIGATE  long               2              105   
                             medium             1              343   
                             short              3               44   
                   RETRIEVE  long               1              788   
                             medium            13              170   
                             short              1              259   
gitlab,reddit      MUTATE    long              10              552   
gitlab,wikipedia   MUTATE    long               6              556   
map,shopping_admin NAVIGATE  long               2              759   
map,wikipedia      RETRIEVE  long               1               97   
reddit             MUTATE    long               8              580   
                             medium            21              400   
                             short              7              399   
                   RETRIEVE  long               5               28   
                             medium             1               66   
reddit,gitlab      MUTATE    long               3              681   
                             medium             5              684   
shopping           MUTATE    long               5              507   
                             medium            13              431   
                             short              3              519   
                   NAVIGATE  medium             9              240   
                             short              1              269   
                   RETRIEVE  long              13               50   
                             medium             7               21   
                             short              5              148   
shopping,reddit    MUTATE    long               5              671   
shopping_admin     MUTATE    long               7              544   
                             medium            11              488   
                             short              8              499   
                   NAVIGATE  medium             2              707   
                             short              4              157   
                   RETRIEVE  long               8              108   
                             medium            11               11   
                             short              4               42   
wikipedia,map      NAVIGATE  long              12              424   
                   RETRIEVE  long               4              265   

                                                                                                                                                                                                                 example_intent  
site               task_type intent_bucket                                                                                                                                                                                       
gitlab             MUTATE    long           Reply on the merge request assigned to me for verification functions: if the last comment is from the author, reply "Thank you"; otherwise tag the author (e.g., @user) as a rem...  
                             medium                                                                                                                       Assign the issue regarding 404 errors in a11yproject to Roshanjossey.  
                             short                                                                                                                                                                                Fork MetaSeq.  
                   NAVIGATE  long          

,task_id,hard_subset,site,task_type,intent_bucket,gym_id,intent
2,444,True,gitlab,MUTATE,long,browsergym/webarena_verified.308.444.2,Update and commit (to a new branch called title-update with no merged request) the website code for the current project using the simple online file editor to change the browse...
1,800,True,gitlab,MUTATE,medium,browsergym/webarena_verified.600.800.2,"create a new group ""x-lab"" with members JonasVautherin, dilipchandima, dawiss1337, bmyun, DCMJY"
0,522,True,gitlab,MUTATE,short,browsergym/webarena_verified.352.522.2,Fork all repos from facebook.
5,28,True,reddit,RETRIEVE,long,browsergym/webarena_verified.33.28.2,"In the Worcester forum, get the username and post title of the most recent post, and count the number of comments on that post that are not from the author and have more downvo..."
4,644,True,reddit,MUTATE,medium,browsergym/webarena_verified.16.644.2,"Post a notice in games forum titled ""Tears of Kingdom Meet up!"". Set post details to ""virtual meetup for Tears of Kingdom on Dec 15th"""
3,407,True,reddit,MUTATE,short,browsergym/webarena_verified.22.407.2,Upvote the newest post in the deep learning forum
8,507,True,shopping,MUTATE,long,browsergym/webarena_verified.172.507.2,Buy the highest rated product from the Ceiling light category within a budget above 1000. Discard any items in your cart if it is not empty.
7,795,True,shopping,MUTATE,medium,browsergym/webarena_verified.191.795.2,"Change the delivery address for my second most recent order to 6726 McPherson Blvd, Pittsburgh, PA."
6,387,True,shopping,RETRIEVE,short,browsergym/webarena_verified.1356.387.2,Who gave 4 or 5 stars for phone cases from EYZUTAK
11,108,True,shopping_admin,RETRIEVE,long,browsergym/webarena_verified.270.108.2,"Get the monthly count of completed orders from January 2023 through May 2023, inclusive. Return a list of objects with keys ""month"" (month name) and ""count"" (as integer) only, ..."


,task_id,site,task_type,intent_bucket,gym_id,intent
397,397,gitlab,MUTATE,short,browsergym/webarena_verified.352.397.2,Fork MetaSeq.
398,398,gitlab,MUTATE,short,browsergym/webarena_verified.352.398.2,Fork all repos from Akilesh Kannan.
44,44,gitlab,NAVIGATE,short,browsergym/webarena_verified.303.44.2,Open my todos page
105,105,gitlab,NAVIGATE,long,browsergym/webarena_verified.349.105.2,Navigate to the page showing the list of not yet closed issues in the OpenAPITools/openapi-generator repository that have labels related to OpenAPI Generator CLI
170,170,gitlab,RETRIEVE,medium,browsergym/webarena_verified.289.170.2,Get the project ID(s) of my personal project(s) that received the least stars
171,171,gitlab,RETRIEVE,medium,browsergym/webarena_verified.289.171.2,Get the project ID(s) of my personal project(s) that received less than 5 stars
552,552,"gitlab,reddit",MUTATE,long,browsergym/webarena_verified.84.552.2,"Use the Web IDE to create a folder named real_space in gimmiethat.space repo. Within it, create a file named urls.json that contains the full URLs of the 5 most recent posts fr..."
553,553,"gitlab,reddit",MUTATE,long,browsergym/webarena_verified.84.553.2,"Use the Web IDE to create a folder named news in gimmiethat.space repo. Within it, create a file named urls.json that contains the full URLs of the 5 most recent posts from the..."
556,556,"gitlab,wikipedia",MUTATE,long,browsergym/webarena_verified.87.556.3,Create a repository named nolan_honest_fans with a README file containing only Christopher Nolan's theatrically released feature-length films (use the provided wiki site to loo...
557,557,"gitlab,wikipedia",MUTATE,long,browsergym/webarena_verified.87.557.3,Create a repository named nolan_old_fans with a README file containing only Christopher Nolan's theatrically released feature-length films before 2010 (use the provided wiki si...


## Smoke Command

In [25]:

if CLEAN_PREVIOUS_EXPERIMENT:
    target = RUN_ROOT / EXPERIMENT_NAME
    if target.exists():
        shutil.rmtree(target)
        print('Deleted previous experiment:', target)

SMOKE_COMMAND = [
    'uv', 'run', 'python', 'scripts/run_hk_agent_experiment.py',
    '--experiment-name', EXPERIMENT_NAME,
    '--task-ids', *SMOKE_TASKS,
    '--hs', *SMOKE_HS,
    '--ks', *SMOKE_KS,
    '--run-mode', 'agent',
    '--planner-model', PLANNER_MODEL,
    '--executor-model', EXECUTOR_MODEL,
    '--max-planner-calls', '3',
    '--max-steps', '8',
    '--llm-timeout-seconds', '600',
]
if ALLOW_NON_HARD_TASK_IDS:
    SMOKE_COMMAND.insert(SMOKE_COMMAND.index('--hs'), '--allow-non-hard-task-ids')

print('$ ' + ' '.join(SMOKE_COMMAND))


$ uv run python scripts/run_hk_agent_experiment.py --experiment-name hk-agent-browsergym-random-gemma4-smoke --task-ids 522 800 444 407 644 28 387 795 507 505 15 108 --hs 0 2 5 --ks 0 2 5 --run-mode agent --planner-model gemma4:26b --executor-model gemma4:e4b --max-planner-calls 3 --max-steps 8 --llm-timeout-seconds 600


In [13]:
if RUN_SMOKE:
    proc = subprocess.Popen(
        SMOKE_COMMAND,
        cwd=ROOT,
        env={**os.environ, **SMOKE_ENV},
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f'Smoke failed with return code {code}')
else:
    print('Run disabled. Set RUN_SMOKE = True to execute the smoke.')


Run disabled. Set RUN_SMOKE = True to execute the smoke.


## Summary Analysis

In [14]:
SUMMARY_PATH = RUN_ROOT / EXPERIMENT_NAME / 'summary.json'
if not SUMMARY_PATH.exists():
    print('Missing summary:', SUMMARY_PATH)
    df = pd.DataFrame()
else:
    summary = json.loads(SUMMARY_PATH.read_text())
    df = pd.DataFrame(summary.get('rows', []))
    display(df)


,task_id,intent_template_id,revision,gym_id,site,sites,h,k,run_mode,planner_model,executor_model,status,official_score,official_success,official_eval_status,runtime_progress_score,runtime_replans,runtime_no_progress_events,runtime_invalid_actions,runtime_loop_events,total_steps,total_runtime_ms,total_tokens,planner_tokens,executor_tokens,num_plan_subgoals_generated,planner_calls,executor_calls,output_dir
0,522,352,2,browsergym/webarena_verified.352.522.2,gitlab,gitlab,0,0,agent,gemma4:26b,gemma4:e4b,completed,0.0,False,failure,0.0,3,3,3,2,3,188810,10963,10963,0,6,3,0,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522


In [15]:
if df.empty:
    print('No rows available.')
else:
    numeric_cols = ['official_score', 'total_tokens', 'planner_tokens', 'executor_tokens', 'total_runtime_ms', 'runtime_replans', 'runtime_no_progress_events']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in ['planner_tokens', 'executor_tokens', 'total_tokens', 'total_runtime_ms', 'runtime_replans', 'runtime_no_progress_events']:
        if col not in df.columns:
            df[col] = 0
    if 'official_success' not in df.columns:
        df['official_success'] = False
    df['official_success_bool'] = df['official_success'].fillna(False).astype(bool)
    df['runtime_seconds'] = df.get('total_runtime_ms', 0) / 1000
    df['planner_cost_proxy'] = df.get('planner_tokens', 0).fillna(0) / 1000 * PLANNER_COST_PER_1K
    df['executor_cost_proxy'] = df.get('executor_tokens', 0).fillna(0) / 1000 * EXECUTOR_COST_PER_1K
    df['total_cost_proxy'] = df['planner_cost_proxy'] + df['executor_cost_proxy']
    success_numeric = df['official_success_bool'].astype(float)
    df['utility'] = success_numeric - LAMBDA_COST * df['total_cost_proxy'] - LAMBDA_RUNTIME_SECONDS * df['runtime_seconds']
    df['validation_mode'] = df['k'].apply(lambda value: 'no_runtime_validation' if int(value) == 0 else f'every_{int(value)}_actions')

    cols = ['task_id', 'h', 'k', 'validation_mode', 'official_score', 'official_success_bool', 'num_plan_subgoals_generated', 'total_tokens', 'planner_tokens', 'executor_tokens', 'total_cost_proxy', 'utility', 'total_runtime_ms', 'runtime_replans', 'runtime_no_progress_events', 'output_dir']
    display(df[[c for c in cols if c in df.columns]].sort_values(['task_id', 'h', 'k']))
    grouped = df.groupby(['h', 'k'], dropna=False).agg(
        official_success_rate=('official_success_bool', 'mean'),
        mean_score=('official_score', 'mean'),
        mean_tokens=('total_tokens', 'mean'),
        mean_planner_tokens=('planner_tokens', 'mean'),
        mean_executor_tokens=('executor_tokens', 'mean'),
        mean_cost_proxy=('total_cost_proxy', 'mean'),
        mean_utility=('utility', 'mean'),
        mean_runtime_ms=('total_runtime_ms', 'mean'),
        mean_replans=('runtime_replans', 'mean'),
    )
    grouped['official_success_percent'] = grouped['official_success_rate'] * 100
    display(grouped)


,task_id,h,k,validation_mode,official_score,official_success_bool,num_plan_subgoals_generated,total_tokens,planner_tokens,executor_tokens,total_cost_proxy,utility,total_runtime_ms,runtime_replans,runtime_no_progress_events,output_dir
0,522,0,0,no_runtime_validation,0.0,False,6,10963,10963,0,0.0,0.0,188810,3,3,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522


,,official_success_rate,mean_score,mean_tokens,mean_planner_tokens,mean_executor_tokens,mean_cost_proxy,mean_utility,mean_runtime_ms,mean_replans,official_success_percent
h,k,,,,,,,,,,
0,0,0.0,0.0,10963.0,10963.0,0.0,0.0,0.0,188810.0,3.0,0.0


## Official Score Breakdown

Dieser Block zaehlt fehlgeschlagene oder abgebrochene Runs als `False`, damit Crashs und Timeouts nicht aus der Erfolgsrate herausfallen. Die Spalten `site_meta`, `task_type` und `intent_bucket` kommen aus den WebArena-Verified Task-Metadaten.


In [16]:
if df.empty:
    print('No rows available.')
else:
    meta_cols = ['task_id', 'hard_subset', 'site', 'task_type', 'intent_bucket', 'intent_length', 'intent']
    score_df = df.merge(task_meta_df[meta_cols], on='task_id', how='left', suffixes=('', '_meta'))
    score_df['official_success_bool'] = score_df.get('official_success_bool', score_df.get('official_success', False)).fillna(False).astype(bool)
    score_df['official_score_filled'] = pd.to_numeric(score_df.get('official_score', 0), errors='coerce').fillna(0)
    score_df['run_failed'] = score_df['status'].ne('completed') if 'status' in score_df.columns else False

    def score_table(group_cols):
        grouped = score_df.groupby(group_cols, dropna=False).agg(
            runs=('task_id', 'count'),
            tasks=('task_id', 'nunique'),
            successes=('official_success_bool', 'sum'),
            failures=('run_failed', 'sum'),
            success_rate=('official_success_bool', 'mean'),
            mean_score=('official_score_filled', 'mean'),
            mean_tokens=('total_tokens', 'mean'),
            mean_runtime_seconds=('runtime_seconds', 'mean'),
            mean_replans=('runtime_replans', 'mean'),
        ).reset_index()
        grouped['success_percent'] = (grouped['success_rate'] * 100).round(1)
        grouped['mean_score'] = grouped['mean_score'].round(3)
        return grouped.sort_values(group_cols)

    overall = pd.DataFrame([{
        'runs': len(score_df),
        'tasks': score_df['task_id'].nunique(),
        'successes': int(score_df['official_success_bool'].sum()),
        'failed_or_crashed_runs': int(score_df['run_failed'].sum()),
        'success_percent': round(float(score_df['official_success_bool'].mean() * 100), 1),
        'mean_score': round(float(score_df['official_score_filled'].mean()), 3),
    }])
    display(overall)

    print('By H/k')
    display(score_table(['h', 'k', 'validation_mode']))

    print('By site')
    display(score_table(['site_meta']))

    print('By task type and length bucket')
    display(score_table(['task_type', 'intent_bucket']))

    print('By site and length bucket')
    display(score_table(['site_meta', 'intent_bucket']))

    print('By task')
    task_cols = ['task_id', 'site_meta', 'task_type', 'intent_bucket']
    display(score_table(task_cols))


,runs,tasks,successes,failed_or_crashed_runs,success_percent,mean_score
0,1,1,0,0,0.0,0.0


By H/k


,h,k,validation_mode,runs,tasks,successes,failures,success_rate,mean_score,mean_tokens,mean_runtime_seconds,mean_replans,success_percent
0,0,0,no_runtime_validation,1,1,0,0,0.0,0.0,10963.0,188.81,3.0,0.0


By site


,site_meta,runs,tasks,successes,failures,success_rate,mean_score,mean_tokens,mean_runtime_seconds,mean_replans,success_percent
0,gitlab,1,1,0,0,0.0,0.0,10963.0,188.81,3.0,0.0


...

,task_type,intent_bucket,runs,tasks,successes,failures,success_rate,mean_score,mean_tokens,mean_runtime_seconds,mean_replans,success_percent
0,MUTATE,short,1,1,0,0,0.0,0.0,10963.0,188.81,3.0,0.0


By site and length bucket


,site_meta,intent_bucket,runs,tasks,successes,failures,success_rate,mean_score,mean_tokens,mean_runtime_seconds,mean_replans,success_percent
0,gitlab,short,1,1,0,0,0.0,0.0,10963.0,188.81,3.0,0.0


By task


,task_id,site_meta,task_type,intent_bucket,runs,tasks,successes,failures,success_rate,mean_score,mean_tokens,mean_runtime_seconds,mean_replans,success_percent
0,522,gitlab,MUTATE,short,1,1,0,0,0.0,0.0,10963.0,188.81,3.0,0.0


## Cost And Utility Analysis

`planner_cost_proxy = planner_tokens / 1000 * PLANNER_COST_PER_1K`; `executor_cost_proxy = executor_tokens / 1000 * EXECUTOR_COST_PER_1K`. For local Ollama this can stay zero. For a thesis cost proxy, set nonzero model-specific values in the smoke configuration cell.

In [17]:
if df.empty:
    print('No rows available.')
else:
    cost_cols = ['task_id', 'h', 'k', 'validation_mode', 'official_success_bool', 'official_score', 'planner_tokens', 'executor_tokens', 'planner_cost_proxy', 'executor_cost_proxy', 'total_cost_proxy', 'runtime_seconds', 'utility']
    display(df[[c for c in cost_cols if c in df.columns]].sort_values(['task_id', 'h', 'k']))

    tradeoff = df.groupby(['h', 'k', 'validation_mode'], dropna=False).agg(
        runs=('task_id', 'count'),
        success_rate=('official_success_bool', 'mean'),
        mean_score=('official_score', 'mean'),
        mean_planner_tokens=('planner_tokens', 'mean'),
        mean_executor_tokens=('executor_tokens', 'mean'),
        mean_total_tokens=('total_tokens', 'mean'),
        mean_cost_proxy=('total_cost_proxy', 'mean'),
        mean_runtime_seconds=('runtime_seconds', 'mean'),
        mean_replans=('runtime_replans', 'mean'),
        mean_utility=('utility', 'mean'),
    ).sort_index()
    display(tradeoff)


,task_id,h,k,validation_mode,official_success_bool,official_score,planner_tokens,executor_tokens,planner_cost_proxy,executor_cost_proxy,total_cost_proxy,runtime_seconds,utility
0,522,0,0,no_runtime_validation,False,0.0,10963,0,0.0,0.0,0.0,188.81,0.0


,,,runs,success_rate,mean_score,mean_planner_tokens,mean_executor_tokens,mean_total_tokens,mean_cost_proxy,mean_runtime_seconds,mean_replans,mean_utility
h,k,validation_mode,,,,,,,,,,
0,0,no_runtime_validation,1,0.0,0.0,10963.0,0.0,10963.0,0.0,188.81,3.0,0.0


## Example Run Rows And Artifacts

In [18]:
if df.empty:
    print('No run rows available.')
else:
    example_run_rows = df.sort_values(['task_id', 'h', 'k']).groupby('task_id', dropna=False).head(1).copy()
    display_cols = ['task_id', 'h', 'k', 'status', 'official_score', 'official_success', 'total_steps', 'num_plan_subgoals_generated', 'total_tokens', 'output_dir']
    display(example_run_rows[[c for c in display_cols if c in example_run_rows.columns]])

    for _, row in example_run_rows.iterrows():
        output_dir = Path(row['output_dir'])
        print('\n' + '=' * 80)
        print('task', row['task_id'], 'H=', row['h'], 'k=', row['k'])
        for name in ['run_summary.json', 'plan.json', 'step_trace.jsonl', 'planner_calls.jsonl', 'executor_calls.jsonl', 'runtime_evaluator_signals.jsonl', 'controller_decisions.jsonl', 'agent_response.json', 'eval_result.json', 'network.har']:
            path = output_dir / name
            print(f'{name:32s} exists={path.exists()} path={path}')

        plan_path = output_dir / 'plan.json'
        if plan_path.exists():
            plan = json.loads(plan_path.read_text())
            subgoals = plan.get('subgoals', [])
            print('subgoals:', len(subgoals))
            for subgoal in subgoals[:5]:
                print('-', subgoal.get('id'), subgoal.get('objective'), '=>', subgoal.get('expected_outcome'))


,task_id,h,k,status,official_score,official_success,total_steps,num_plan_subgoals_generated,total_tokens,output_dir
0,522,0,0,completed,0.0,False,3,6,10963,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522



task 522 H= 0 k= 0
run_summary.json                 exists=True path=/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522/run_summary.json
plan.json                        exists=True path=/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522/plan.json
step_trace.jsonl                 exists=True path=/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522/step_trace.jsonl
planner_calls.jsonl              exists=True path=/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-random-gemma4-smoke/gitlab/522/h0_k0/522/planner_calls.jsonl
executor_calls.jsonl             exists=False path=/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browser

## Task And Subgoal Category Analysis

In [19]:
if df.empty:
    print('No run rows available.')
else:
    meta_cols = ['task_id', 'hard_subset', 'site', 'task_type', 'intent_length', 'intent_bucket', 'gym_id', 'intent']
    analysis_df = df.merge(task_meta_df[meta_cols], on='task_id', how='left', suffixes=('', '_meta'))
    for metric in ['num_plan_subgoals_generated', 'planner_calls', 'executor_calls', 'runtime_replans', 'total_tokens', 'official_score', 'official_success_bool']:
        if metric not in analysis_df.columns:
            analysis_df[metric] = pd.NA
    display(analysis_df[[c for c in ['task_id', 'hard_subset', 'site_meta', 'task_type', 'intent_bucket', 'h', 'k', 'num_plan_subgoals_generated', 'planner_calls', 'executor_calls', 'official_success_bool', 'official_score', 'total_tokens', 'runtime_replans'] if c in analysis_df.columns]])

    category_summary = analysis_df.groupby(['hard_subset', 'site_meta', 'task_type', 'intent_bucket'], dropna=False).agg(
        tasks=('task_id', 'nunique'),
        runs=('task_id', 'count'),
        mean_subgoals=('num_plan_subgoals_generated', 'mean'),
        mean_planner_calls=('planner_calls', 'mean'),
        mean_executor_calls=('executor_calls', 'mean'),
        mean_replans=('runtime_replans', 'mean'),
        mean_tokens=('total_tokens', 'mean'),
        success_rate=('official_success_bool', 'mean'),
        mean_score=('official_score', 'mean'),
    ).sort_index()
    display(category_summary)

    hk_category_summary = analysis_df.groupby(['h', 'k', 'task_type', 'intent_bucket'], dropna=False).agg(
        runs=('task_id', 'count'),
        mean_subgoals=('num_plan_subgoals_generated', 'mean'),
        mean_tokens=('total_tokens', 'mean'),
        success_rate=('official_success_bool', 'mean'),
        mean_score=('official_score', 'mean'),
    ).sort_index()
    display(hk_category_summary)


,task_id,hard_subset,site_meta,task_type,intent_bucket,h,k,num_plan_subgoals_generated,planner_calls,executor_calls,official_success_bool,official_score,total_tokens,runtime_replans
0,522,True,gitlab,MUTATE,short,0,0,6,3,0,False,0.0,10963,3


,,,,tasks,runs,mean_subgoals,mean_planner_calls,mean_executor_calls,mean_replans,mean_tokens,success_rate,mean_score
hard_subset,site_meta,task_type,intent_bucket,,,,,,,,,
True,gitlab,MUTATE,short,1,1,6.0,3.0,0.0,3.0,10963.0,0.0,0.0


,,,,runs,mean_subgoals,mean_tokens,success_rate,mean_score
h,k,task_type,intent_bucket,,,,,
0,0,MUTATE,short,1,6.0,10963.0,0.0,0.0
